# Argentina SYNOP cloud data retrieval

Starter notebook for **cloudVis**: download surface synoptic (SYNOP) observations over Argentina and extract cloud-related fields.

**Terminology note**
- **Cloud cover** (`Nt` in decoded data): sky fraction in **oktas** (0–8 eighths), not feet.
- **Cloud base height** (`HKm` in decoded data): lowest cloud base in **kilometers** above ground. Convert to feet with `HKm * 3280.84`.

If you meant cloud *cover*, use `Nt`. If you meant cloud *base altitude*, use `HKm` and convert to feet.

**Data sources used here**
1. [OGIMET](http://www.ogimet.com/) — raw SYNOP bulletins (free, no API key)
2. [`atmofetch`](https://pypi.org/project/atmofetch/) — decoded hourly SYNOP with station coordinates

Later steps (not in this notebook yet): map plotting and spatial interpolation over Argentina.

In [ ]:
# Run this cell first in Google Colab
!pip install -q atmofetch synop2bufr pandas requests

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from io import StringIO
from typing import Iterable

import pandas as pd
import requests
from atmofetch import meteo_ogimet, stations_ogimet
from synop2bufr import parse_synop

# --- user configuration ---
END_UTC = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
START_UTC = END_UTC - timedelta(hours=6)  # keep short for Colab; increase once it works

KM_TO_FEET = 3280.84
OGIMET_URL = "http://www.ogimet.com/cgi-bin/getsynop"

## 1. List Argentina SYNOP stations (with coordinates)

Argentina WMO block is **87xxx** (South America).

In [ ]:
stations = stations_ogimet(country="Argentina")
stations = stations.rename(columns={"wmo_id": "station_id", "lon": "lon", "lat": "lat"})
print(f"{len(stations)} stations found")
stations.head(10)

## 2. Download decoded hourly SYNOP (recommended starting point)

This uses OGIMET behind the scenes and returns a tidy table. OGIMET rate-limits requests, so we start with a handful of stations.

In [ ]:
def fetch_decoded_synop(
    station_ids: Iterable[int | str],
    start: datetime,
    end: datetime,
) -> pd.DataFrame:
    """Download decoded hourly SYNOP for one or more WMO stations."""
    df = meteo_ogimet(
        interval="hourly",
        station=list(station_ids),
        date_range=(start.date().isoformat(), end.date().isoformat()),
        coords=True,
    )
    df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
    mask = (df["Date"] >= pd.Timestamp(start)) & (df["Date"] <= pd.Timestamp(end))
    return df.loc[mask].copy()


sample_station_ids = stations["station_id"].head(8).tolist()
decoded = fetch_decoded_synop(sample_station_ids, START_UTC, END_UTC)

cloud_cols = [c for c in ["Nt", "Nh", "HKm"] if c in decoded.columns]
decoded["cloud_base_ft"] = decoded["HKm"] * KM_TO_FEET if "HKm" in decoded.columns else pd.NA

preview_cols = ["station_ID", "Date", "Lon", "Lat", *cloud_cols, "cloud_base_ft", "Viskm"]
preview_cols = [c for c in preview_cols if c in decoded.columns]
decoded[preview_cols].dropna(subset=["HKm"], how="all").head(20)

## 3. Download raw SYNOP bulletins for all Argentina (bulk)

Useful when you want full control over decoding or need fields not exposed by `atmofetch`.

In [ ]:
def download_raw_synop_argentina(start: datetime, end: datetime) -> pd.DataFrame:
    params = {
        "begin": start.strftime("%Y%m%d%H%M"),
        "end": end.strftime("%Y%m%d%H%M"),
        "state": "Argen",  # OGIMET matches countries by prefix
        "header": "yes",
        "lang": "eng",
    }
    response = requests.get(OGIMET_URL, params=params, timeout=120)
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text))


raw = download_raw_synop_argentina(START_UTC, END_UTC)
raw = raw.rename(
    columns={
        "WMO_ID": "station_id",
        "ANO": "year",
        "MES": "month",
        "DIA": "day",
        "HORA": "hour",
        "MINUTO": "minute",
        "PARTE": "report",
    }
)
print(f"{len(raw)} raw SYNOP messages")
raw.head()

## 4. Decode raw SYNOP messages (cloud cover + lowest cloud base)

Not every message contains a cloud-base group. `lowest_cloud_base` is in **meters** when present.

In [ ]:
def decode_synop_row(row: pd.Series) -> dict | None:
    try:
        decoded_msg, *_ = parse_synop(row["report"], int(row["year"]), int(row["month"]))
    except Exception:
        return None

    base_m = decoded_msg.get("lowest_cloud_base")
    return {
        "station_id": decoded_msg.get("station_id"),
        "datetime_utc": pd.Timestamp(
            year=int(row["year"]),
            month=int(row["month"]),
            day=int(row["day"]),
            hour=int(row["hour"]),
            minute=int(row["minute"]),
            tz="UTC",
        ),
        "cloud_cover_oktas": decoded_msg.get("cloud_cover"),
        "cloud_base_m": base_m,
        "cloud_base_ft": base_m * 3.28084 if base_m is not None else None,
    }


decoded_from_raw = pd.DataFrame(
    [row for row in (decode_synop_row(r) for _, r in raw.iterrows()) if row is not None]
)

decoded_from_raw = decoded_from_raw.merge(
    stations[["station_id", "lat", "lon", "station_names"]],
    on="station_id",
    how="left",
)

decoded_from_raw.sort_values("datetime_utc").head(20)

## 5. Quick sanity check: stations reporting cloud base height

In [ ]:
with_base = decoded_from_raw.dropna(subset=["cloud_base_ft"])
print(f"Messages with cloud base height: {len(with_base)} / {len(decoded_from_raw)}")

if not with_base.empty:
    display(
        with_base[["datetime_utc", "station_id", "station_names", "lat", "lon", "cloud_cover_oktas", "cloud_base_ft"]]
        .sort_values("cloud_base_ft")
        .head(15)
    )
else:
    print("No cloud base height in this time window. Try a longer range or use decoded['HKm'] from section 2.")

## 6. Optional preview map (station points only)

Interpolation over the full territory comes later. This just plots observed station values.

In [ ]:
!pip install -q folium

import folium

map_df = decoded.dropna(subset=["HKm", "Lat", "Lon"]).copy()
if map_df.empty:
    map_df = with_base.copy() if "with_base" in globals() and not with_base.empty else pd.DataFrame()
    value_col = "cloud_base_ft"
    lat_col, lon_col = "lat", "lon"
else:
    value_col = "cloud_base_ft"
    lat_col, lon_col = "Lat", "Lon"

if map_df.empty:
    print("No cloud-base observations to map in the selected window.")
else:
    m = folium.Map(location=[-38.5, -63.5], zoom_start=4, tiles="CartoDB positron")
    for _, row in map_df.iterrows():
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=6,
            popup=f"Station {row.get('station_ID', row.get('station_id'))}<br>Cloud base: {row[value_col]:.0f} ft",
            color="#2563eb",
            fill=True,
            fill_opacity=0.8,
        ).add_to(m)
    m

## Next steps

1. **Scale up stations**: loop over all `stations['station_id']` in batches (OGIMET pauses ~20 s between monthly chunks).
2. **Pick one timestamp**: filter to a single UTC hour before mapping/interpolating.
3. **Interpolation**: `scipy.interpolate.griddata` or `pykrige` on a lat/lon grid clipped to Argentina's border (GeoJSON from Natural Earth).
4. **SMN official data**: Argentina's Servicio Meteorológico Nacional also publishes observations via WIS2/OGC APIs, but access is less documented than OGIMET.

Share this notebook from Colab via *File → Share* so coworkers can run it in their browsers.